# 00 — Model Hierarchy Tour

Verify the codebase loads, instantiate every model at a common parameter set, and confirm the §9.3 consistency identities hold numerically.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from steering.params import ModelParams, ForcingParams
from steering.models import (
    DuffingModel,
    BesselSteeringModel,
    ContinuousPFLModel,
    DiscretePFLModel,
    FullCircuitModel,
)
from steering.dynamics import VelocityDynamics, AccelerationDynamics
from steering.integrator import Simulation
from steering.visualization.style import use_paper_style

use_paper_style()


## Common parameters

We pick $\kappa_h = \kappa_g = 3$, $\delta = \pi/4$, $S = A = 1$, $\Delta = 3\pi/8$ (the *Drosophila* PFL3 phase shift).

In [ ]:
params = ModelParams(kappa_h=3.0, kappa_g=3.0, delta=np.pi/4)
params_disc8 = params.replace(N_neurons=8)
params_disc64 = params.replace(N_neurons=64)

duffing = DuffingModel.from_params(params)
bessel = BesselSteeringModel()
cont = ContinuousPFLModel(n_quad=128)
disc = DiscretePFLModel()
full = FullCircuitModel()
print('c1, c3 =', bessel.taylor_coefficients(params))


## $U(\theta)$ across all five models

Bessel and ContinuousPFL with the quadratic nonlinearity should overlap exactly. Duffing should match near $\theta=0$ and deviate at the boundaries. DiscretePFL with $N=64$ should be visually indistinguishable from the continuous model; $N=8$ shows discretization artifacts.

In [ ]:
theta = np.linspace(-np.pi, np.pi, 401)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(theta, bessel.steering_drive(theta, params), label='Bessel', lw=2)
ax.plot(theta, cont.steering_drive(theta, params), label='ContinuousPFL', ls=':', lw=2)
ax.plot(theta, disc.steering_drive(theta, params_disc64), label='DiscretePFL N=64', ls='--')
ax.plot(theta, disc.steering_drive(theta, params_disc8), label='DiscretePFL N=8', alpha=0.7)
ax.plot(theta, duffing.steering_drive(theta), label='Duffing (cubic)', ls='-.', lw=1.2)
ax.axhline(0, color='0.6', lw=0.5)
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$U(\theta)$')
ax.legend(); plt.show()


## Consistency table (§9.3 identity chains)

In [ ]:
grid = np.linspace(-np.pi+0.05, np.pi-0.05, 21)
Ub = np.asarray(bessel.steering_drive(grid, params))
Uc = np.asarray(cont.steering_drive(grid, params))
Ud = duffing.steering_drive(grid)
print(f'max|Bessel - Continuous|        = {np.max(np.abs(Ub-Uc)):.2e}')
print(f'max|Duffing - Bessel| / |Bessel| = {np.max(np.abs(Ud-Ub))/np.max(np.abs(Ub)):.2e}')
Ud256 = disc.steering_drive(grid, params.replace(N_neurons=256))
print(f'max|Continuous - Discrete N=256| = {np.max(np.abs(Uc-Ud256)):.2e}')
full_w0 = full.steering_drive(grid, params)
print(f'max|FullCircuit(W=0) - Continuous| = {np.max(np.abs(full_w0-Uc)):.2e}')


## Effect of nonlinearity

Pick a bistable parameter set; compare $U(\theta)$ for $f \in \{x^2, \mathrm{ELU}, \mathrm{ReLU}, \mathrm{softplus}\}$ using the continuous model. The Bessel closed form is *only* exact for $f=x^2$.

In [ ]:
p_bi = ModelParams(kappa_h=2.0, kappa_g=2.0, delta=1.4)
theta = np.linspace(-np.pi, np.pi, 401)
fig, ax = plt.subplots(figsize=(7, 4))
for nl in ['quadratic', 'elu', 'relu', 'softplus']:
    U = ContinuousPFLModel(n_quad=128).steering_drive(theta, p_bi.replace(nonlinearity=nl))
    ax.plot(theta, U / np.max(np.abs(U) + 1e-12), label=nl)
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$U(\theta)$ (normalised)')
ax.legend(); ax.axhline(0, color='0.6', lw=0.5)
plt.show()
